# AG-UI with Strands Agents

Run an AG-UI agent **directly in the notebook**.

You'll:
1. Install dependencies
2. Define a custom tool (`get_weather`)
3. Create a Strands agent and run the agent directly (simple call)
4. Wrap with AG-UI and stream events interactively

## 1. Install Dependencies

In [ ]:
#!pip install strands-agents ag-ui-strands nest-asyncio

## 2. Define a Custom Tool

Strands uses the `@tool` decorator to turn any Python function into a tool the LLM can call.
The docstring becomes the tool description, and type hints define the parameter schema.

In [ ]:
from strands import Agent, tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city.

    Args:
        city: The name of the city to get weather for.

    Returns:
        A string describing the current weather.
    """
    weather_data = {
        "paris": "Cloudy with light winds",
        "london": "Rainy",
        "new york": "Sunny and clear",
        "tokyo": "Partly cloudy",
    }
    result = weather_data.get(city.lower(), f"Pleasant weather in {city}")
    return f"Weather in {city}: {result}"

print("Tool defined: get_weather")

## 3. Create the Strands Agent

The agent uses Amazon Bedrock as the model provider by default.
We register our `get_weather` tool so the LLM can call it. Run the Agent Directly (no AG-UI)

In [ ]:
strands_agent = Agent(
    system_prompt=(
        "You are a helpful weather assistant. "
        "Use the get_weather tool for current conditions in a city. "
        "Always provide a friendly, concise response. "
        "Do NOT use any emoji, emoticons, icons, or non-ASCII symbols "
        "in your responses. Use plain text only."
    ),
    tools=[get_weather],
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0"
)

print("Strands agent created")

response = strands_agent("What's the weather in Paris?")
print(response)

## 4. Wrap with AG-UI and Stream Events

Now let's add the AG-UI protocol layer. This wraps the same agent so it emits **typed AG-UI events** instead of a plain string response.

This is what a frontend (CopilotKit, React, Slack, etc.) would consume.

In [ ]:
import asyncio
import nest_asyncio
from uuid import uuid4
from ag_ui_strands import StrandsAgent
from ag_ui.core import RunAgentInput

# Allow nested event loops in Jupyter
nest_asyncio.apply()

# Wrap the Strands agent with AG-UI protocol
agui_agent = StrandsAgent(
    agent=strands_agent,
    name="weather_agent",
    description="A weather assistant that looks up weather by city",
)

print("AG-UI wrapper created — agent now emits typed events")

### Raw Event Stream

Let's see every AG-UI event as a raw Python object.
This is what flows over the wire (SSE/WebSocket) to any AG-UI client.

In [ ]:
def strip_non_ascii(text: str) -> str:
    """Drop emoji / non-ASCII so Jupyter's serializer never hits a surrogate."""
    if not isinstance(text, str):
        text = str(text)
    return text.encode("ascii", errors="ignore").decode("ascii")


async def print_raw_events(message: str):
    """Run the agent and print the major events in a clean, readable way."""

    run_input = RunAgentInput(
        threadId=str(uuid4()),
        runId=str(uuid4()),
        messages=[{"id": str(uuid4()), "role": "user", "content": message}],
        state={},
        tools=[],
        context=[],
        forwardedProps={},
    )

    print(f"You: {message}\n")

    tool_args = ""   # accumulate streaming tool arguments

    async for event in agui_agent.run(run_input):
        name = str(getattr(event, "type", "")).split(".")[-1].upper()

        # --- Agent lifecycle ---
        if name == "RUN_STARTED":
            print("Agent started")

        elif name == "RUN_FINISHED":
            print("Agent stopped")

        # --- Tool call sequence ---
        elif name == "TOOL_CALL_START":
            print(f"  Tool started : {getattr(event, 'toolCallName', '')}")

        elif name == "TOOL_CALL_ARGS":
            tool_args += getattr(event, "delta", "") or ""

        elif name == "TOOL_CALL_END":
            if tool_args:
                print(f"  Tool arguments: {strip_non_ascii(tool_args)}")
                tool_args = ""
            output = getattr(event, "output", None) or getattr(event, "result", None)
            if output:
                print(f"  Tool output  : {strip_non_ascii(str(output))}")
            print("  Tool ended")

        # --- Message sequence ---
        elif name == "TEXT_MESSAGE_START":
            print("\nMessage started")
            print("Answer: ", end="", flush=True)

        elif name == "TEXT_MESSAGE_CONTENT":
            print(strip_non_ascii(getattr(event, "delta", "") or ""), end="", flush=True)

        elif name == "TEXT_MESSAGE_END":
            print()                 # newline to close the streamed answer
            print("Message ended")


asyncio.get_event_loop().run_until_complete(
    print_raw_events("What is the weather in Tokyo?")
)

---

### How This Maps to AG-UI

| What the code prints | AG-UI event it reacts to |
|---------------------|------------------------|
| `Agent started` | `RUN_STARTED` |
| `Tool started : <name>` | `TOOL_CALL_START` |
| `Tool arguments: {...}` | `TOOL_CALL_ARGS` (deltas concatenated) |
| `Tool output : ...` | `TOOL_CALL_END` (output field, if present) |
| `Tool ended` | `TOOL_CALL_END` |
| `Message started` | `TEXT_MESSAGE_START` |
| `Answer: ...` (streamed on one line) | `TEXT_MESSAGE_CONTENT` (deltas concatenated) |
| `Message ended` | `TEXT_MESSAGE_END` |
| `Agent stopped` | `RUN_FINISHED` |

The function walks the raw AG-UI event stream and prints one line per major
milestone — the agent lifecycle (`RUN_STARTED` / `RUN_FINISHED`), each tool
call (`TOOL_CALL_START` → `TOOL_CALL_ARGS` → `TOOL_CALL_END`), and the
assistant message (`TEXT_MESSAGE_START` → `TEXT_MESSAGE_CONTENT` →
`TEXT_MESSAGE_END`). This is exactly the **same event-processing loop** that a
real AG-UI client — CopilotKit, a React app, or a Slack bot — implements. The
protocol is identical regardless of the client; only the rendering differs.